In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu

NL = chr(10)

# 1. Загрузка евразийских CPI и Welfare Ratios (Allen et al., 2011)
xl_ch = pd.ExcelFile('China_1730-1933_Allen_et_al.xls')
d_fig = xl_ch.parse('Figures2-6')

cpi_df = d_fig.iloc[7:, 10:19].copy()
cpi_df.columns = ['year', 'London', 'Amsterdam', 'Leipzig', 'Florence', 'Beijing', 'Suzhou', 'Canton', 'Kyoto_Tokyo']
cpi_df['year'] = pd.to_numeric(cpi_df['year'], errors='coerce')
cpi_df = cpi_df.dropna(subset=['year']).set_index('year').apply(pd.to_numeric, errors='coerce')

wr_df = d_fig.iloc[7:, 21:32].copy()
wr_df.columns = ['year', 'London', 'Amsterdam', 'Leipzig', 'Milan', 'Beijing', 'Shanghai', 'Canton', 'Tokyo', 'Bengal', 'Oxford']
wr_df['year'] = pd.to_numeric(wr_df['year'], errors='coerce')
wr_df = wr_df.dropna(subset=['year']).set_index('year').apply(pd.to_numeric, errors='coerce')

# 2. Загрузка серебряных корзин Москвы (Korchmina, 2025)
xl_k = pd.ExcelFile('Moscow living standards, Korchmina Calculations for Rivista.xlsx')
d_mos = xl_k.parse('subsist for Moscow silver AK')
headers = [str(v).strip() for v in d_mos.head(1).to_numpy().ravel()]
mos_df = pd.DataFrame({
    'year': pd.to_numeric(d_mos.iloc[1:, 0], errors='coerce'),
    'basket_silver_g': pd.to_numeric(d_mos.iloc[1:, headers.index('bare bone basket in silver')], errors='coerce')
}).dropna(subset=['year']).set_index('year')
mos_clean = mos_df.loc[(mos_df.index >= 1738) & (mos_df.index <= 1859), 'basket_silver_g'].dropna()

# 3. Спектральный аудит, граф переходов и бутстрэп (Таблица 1)
def safe_zscore(M_arr):
    mean_v = np.mean(M_arr, axis=0)
    std_v = np.std(M_arr, axis=0, ddof=1)
    safe_std = np.where(std_v > 1e-12, std_v, 1.0)
    Z = (M_arr - mean_v) / safe_std
    Z[:, std_v <= 1e-12] = 0.0
    return Z

def run_eurasian_audit(dlnX, yrs, label, n_boot=300, seed=42):
    Z = safe_zscore(dlnX)
    M, N = Z.shape
    U, s, Vt = np.linalg.svd(Z, full_matrices=False)
    u1, v1, v2 = np.take(U, 0, axis=1), np.take(Vt, 0, axis=0), np.take(Vt, 1, axis=0)
    var1 = (s.item(0)**2) / np.sum(s**2) * 100.0
    var2 = (s.item(1)**2) / np.sum(s**2) * 100.0
    pr_u1 = (1.0 / np.sum(u1**4) - 1.0) / (M - 1.0)
    pr_v1 = (1.0 / np.sum(v1**4) - 1.0) / (N - 1.0)

    rng = np.random.default_rng(seed)
    sub_M = int(np.floor(M * 0.8))
    b_pru, b_prv = [], []
    for _ in range(n_boot):
        idx = np.sort(rng.choice(M, size=sub_M, replace=False))
        Ub, _, Vtb = np.linalg.svd(safe_zscore(dlnX[idx, :]), full_matrices=False)
        u1b, v1b = np.take(Ub, 0, axis=1), np.take(Vtb, 0, axis=0)
        b_pru.append((1.0 / np.sum(u1b**4) - 1.0) / (sub_M - 1.0))
        b_prv.append((1.0 / np.sum(v1b**4) - 1.0) / (N - 1.0))

    dist_mat = squareform(pdist(Z, metric='euclidean'))
    sigma_med = np.median(dist_mat[np.triu_indices(M, k=1)])
    W_full = np.exp(-(dist_mat**2) / (2.0 * sigma_med**2))
    np.fill_diagonal(W_full, 0.0)
    A = np.zeros((M, M))
    for i in range(M):
        for j in np.argsort(dist_mat[i, :])[1:7]:
            A[i, j] = W_full[i, j]
            A[j, i] = W_full[j, i]
    deg = np.sum(A, axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(deg, 1e-12)))
    _, eigvecs = np.linalg.eigh(np.eye(M) - d_inv_sqrt @ A @ d_inv_sqrt)
    psi2 = np.take(eigvecs, 1, axis=1)
    best_p, best_tau = 1.0, -1
    for tau in range(12, M - 12):
        _, pval = mannwhitneyu(psi2[:tau], psi2[tau:], alternative='two-sided')
        if pval < best_p:
            best_p, best_tau = pval, tau

    print(f"{label:<38} | M={M:3d} | Var1={var1:4.1f}% | Var2={var2:4.1f}% | PR(u1)={pr_u1:.3f} [{np.percentile(b_pru, 2.5):.2f}, {np.percentile(b_pru, 97.5):.2f}] | PR(v1)={pr_v1:.3f} [{np.percentile(b_prv, 2.5):.2f}, {np.percentile(b_prv, 97.5):.2f}] | Break={int(yrs.item(best_tau))} (p={best_p:.4f})")
    pvals_scan = [mannwhitneyu(psi2[:tau], psi2[tau:], alternative='two-sided').pvalue for tau in range(12, M - 12)]
    return v1, v2, yrs, pvals_scan

CORE_5 = ['London', 'Amsterdam', 'Leipzig', 'Florence', 'Beijing']
CORE_7 = ['London', 'Amsterdam', 'Leipzig', 'Florence', 'Beijing', 'Suzhou', 'Canton']
WR_5 = ['London', 'Amsterdam', 'Leipzig', 'Milan', 'Beijing']

sub_5 = cpi_df.loc[1738:1913, CORE_5].interpolate(method='linear', limit_direction='both')
dln_5 = np.diff(np.log(sub_5.to_numpy(dtype=float)), axis=0)
yrs_5 = np.arange(1739, 1914)

sub_7_unint = cpi_df.loc[1738:1833, CORE_7]
dln_7_unint = np.diff(np.log(sub_7_unint.to_numpy(dtype=float)), axis=0)
yrs_7_unint = np.arange(1739, 1834)

sub_7_bridged = cpi_df.loc[1738:1913, CORE_7].interpolate(method='linear', limit_direction='both')
dln_7_bridged = np.diff(np.log(sub_7_bridged.to_numpy(dtype=float)), axis=0)

sub_wr5 = wr_df.loc[1738:1913, WR_5].interpolate(method='linear', limit_direction='both')
dln_wr5 = np.diff(np.log(sub_wr5.to_numpy(dtype=float)), axis=0)

print("=== TABLE 1: EURASIAN SPECTRAL SUMMARY ===")
run_eurasian_audit(dln_5, yrs_5, "Unbroken 5-City CPI (1739-1913)")
run_eurasian_audit(dln_7_unint, yrs_7_unint, "Uninterpolated 7-City CPI (1739-1833)")
v1_7, v2_7, _, scan_p7 = run_eurasian_audit(dln_7_bridged, yrs_5, "Bridged 7-City CPI (1739-1913)")
run_eurasian_audit(dln_5[yrs_5 <= 1839], yrs_5[yrs_5 <= 1839], "Pre-Opium War CPI (1739-1839, N=5)")
run_eurasian_audit(dln_5[yrs_5 >= 1840], yrs_5[yrs_5 >= 1840], "Treaty Port & Silver CPI (1840-1913)")
run_eurasian_audit(dln_wr5, yrs_5, "5-City Welfare Ratios (1739-1913)")

# 4. Отрисовка Figure_1_Eurasia.png и Figure_2_Eurasia.png (300 dpi)
v1_7 = -v1_7
yrs_c = np.arange(1738, 1914)

fig1 = plt.figure(figsize=(15.5, 10.5), dpi=300)
plt.subplots_adjust(hspace=0.36, wspace=0.26)

ax1 = fig1.add_subplot(2, 2, 1)
ax1.plot(yrs_c, cpi_df.loc[1738:1913, 'London'], color='#1f4e78', lw=1.6, label='London (Western Maritime Core)')
ax1.plot(yrs_c, cpi_df.loc[1738:1913, 'Amsterdam'], color='#5b9bd5', lw=1.3, label='Amsterdam (North Sea Core)')
ax1.plot(yrs_c, cpi_df.loc[1738:1913, 'Beijing'], color='#c00000', lw=1.8, label='Beijing (Continental Qing North)')
ax1.scatter(mos_clean.index, mos_clean.values, color='#385723', s=24, zorder=5, label='Moscow (Korchmina Silver Basket)')
ax1.axvline(1873, color='gray', linestyle='--', lw=1.0)
ax1.annotate('1873 Global Silver' + NL + 'Demonetization', xy=(1873, 420), xytext=(1832, 680), arrowprops=dict(arrowstyle='->', color='black', lw=0.8), fontsize=8, fontweight='bold')
ax1.set_title("A. Eurasian Living Costs in Grams of Silver per Year (1738-1913)", fontsize=10.5, fontweight='bold')
ax1.set_ylabel("Grams of Pure Silver per Person/Year", fontsize=9.5)
ax1.set_ylim(50, 1050)
ax1.legend(loc='upper left', fontsize=7.8)
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = fig1.add_subplot(2, 2, 2)
dec_labels = ['1720s', '1730s', '1740s', '1750s', '1760s', '1770s', '1780s', '1790s', '1800s']
mos_w = [0.256, 0.229, 0.243, 0.225, 0.287, 0.418, 0.692, 0.713, 0.670]
stp_w = [0.298, 0.366, 0.333, 0.301, 0.410, np.nan, 0.514, 0.457, 0.712]
ndv_w = [0.219, 0.176, 0.211, 0.130, 0.271, 0.295, 0.360, 0.758, 0.599]
x_d = np.arange(len(dec_labels))
ax2.plot(x_d, stp_w, marker='o', color='#1f4e78', lw=1.7, label='Novgorod - St. Petersburg (Baltic)')
ax2.plot(x_d, mos_w, marker='s', color='#c00000', lw=1.7, label='Moscow (Interior Center)')
ax2.plot(x_d, ndv_w, marker='^', color='#385723', lw=1.5, linestyle='--', label='North Dvina / Arkhangelsk (White Sea)')
ax2.set_xticks(x_d)
ax2.set_xticklabels(dec_labels, fontsize=8.5)
ax2.set_ylabel("Grams of Silver per kg of Wheat", fontsize=9.5)
ax2.set_title("B. Internal Russian Grain Price Convergence in Silver (1720s-1800s)", fontsize=10.5, fontweight='bold')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.4)

ax3 = fig1.add_subplot(2, 2, 3)
cities_5 = ['London', 'Amsterdam', 'Florence', 'Leipzig', 'Beijing']
pre_1840 = [36.27, 37.52, 16.84, 5.41, 3.97]
post_1840 = [28.66, 21.23, 23.87, 25.27, 0.97]
x_c = np.arange(len(cities_5))
w_c = 0.34
ax3.bar(x_c - w_c/2, pre_1840, width=w_c, color='#8eaadb', label='Pre-Opium War Era (1739-1839, Var1=31.1%)')
ax3.bar(x_c + w_c/2, post_1840, width=w_c, color='#c00000', label='Treaty Port & Silver Crisis (1840-1913, Var1=59.2%)')
ax3.axhline(20.0, color='black', linestyle=':', lw=1.1, label='Uniform Benchmark (1/5 = 20.0%)')
ax3.text(4 + w_c/2, 2.2, "0.97%", ha='center', fontsize=8, fontweight='bold', color='#800000')
ax3.set_xticks(x_c)
ax3.set_xticklabels(cities_5, fontsize=9)
ax3.set_ylim(0, 44)
ax3.set_ylabel("Share of Eurasian Mode 1 Energy (%)", fontsize=9.5)
ax3.set_title("C. European Monolith vs. Beijing Decoupling (1739-1913)", fontsize=10.5, fontweight='bold')
ax3.legend(loc='upper right', fontsize=7.8)
ax3.grid(True, axis='y', linestyle='--', alpha=0.4)

ax4 = fig1.add_subplot(2, 2, 4)
scan_yrs = [int(yrs_5.item(tau)) for tau in range(12, len(yrs_5) - 12)]
ax4.semilogy(scan_yrs, scan_p7, color='#1f4e78', lw=1.6, label='7-City Eurasian Graph Split p-value')
ax4.axhline(0.05, color='gray', linestyle='--', lw=0.9, label='p = 0.05 Threshold')
ax4.axhline(0.01, color='#c00000', linestyle=':', lw=0.9, label='p = 0.01 Threshold')
best_idx = int(np.argmin(scan_p7))
best_yr = scan_yrs[best_idx]
ax4.scatter([best_yr], [min(scan_p7)], color='#c00000', s=40, zorder=5)
ax4.annotate('1891 Structural Break' + NL + '(p = 0.0008, Silver Crisis)', xy=(best_yr, min(scan_p7)), xytext=(1852, 0.0035), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.9), ha='center', fontsize=8.5, fontweight='bold', color='#800000')
ax4.set_ylabel("Mann-Whitney U p-value (log scale)", fontsize=9.5)
ax4.set_title("D. Spectral Graph Regime Break Detection (1739-1913)", fontsize=10.5, fontweight='bold')
ax4.legend(loc='lower left', fontsize=8)
ax4.grid(True, linestyle='--', alpha=0.4)

fig1.suptitle("Eurasian Continental Bridges: Silver Prices, Internal Russian Convergence, and the Great Divergence (1738-1913)", fontsize=12.2, fontweight='bold', y=0.98)
plt.savefig('Figure_1_Eurasia.png', bbox_inches='tight')
plt.close(fig1)
fig2 = plt.figure(figsize=(15.5, 6.8), dpi=300)
plt.subplots_adjust(wspace=0.26)

ax1 = fig2.add_subplot(1, 2, 1)
offsets_7 = {
    'London': (-0.06, -0.055),
    'Florence': (0.018, -0.035),
    'Canton': (-0.06, 0.03),
    'Suzhou': (-0.06, -0.055)
}
for idx_n, cname in enumerate(CORE_7):
    x_v = float(v1_7.item(idx_n))
    y_v = float(v2_7.item(idx_n))
    col_pt = '#c00000' if cname == 'Beijing' else ('#ed7d31' if cname in ['Suzhou', 'Canton'] else '#1f4e78')
    sz = 110 if cname == 'Beijing' else 75
    ax1.scatter([x_v], [y_v], color=col_pt, s=sz, zorder=5)
    dx, dy = offsets_7.get(cname, (0.018, 0.025))
    ax1.annotate(cname, xy=(x_v, y_v), xytext=(x_v + dx, y_v + dy), fontsize=9, fontweight='bold', color=col_pt)

ax1.axhline(0, color='black', linestyle=':', lw=0.9)
ax1.axvline(0, color='black', linestyle=':', lw=0.9)
ax1.text(0.41, -0.18, 'European Core' + NL + '(London, Amsterdam,' + NL + 'Florence, Leipzig)', fontsize=8.5, fontweight='bold', color='#1f4e78')
ax1.text(0.08, 0.34, 'Continental Qing North' + NL + '(Beijing / Kyakhta Terminus)', fontsize=8.5, fontweight='bold', color='#800000')
ax1.text(-0.34, -0.45, 'Maritime Treaty-Port China' + NL + '(Canton & Lower Yangzi)', fontsize=8.5, fontweight='bold', color='#c55a11')
ax1.set_xlim(-0.40, 0.66)
ax1.set_ylim(-0.76, 0.45)
ax1.set_xlabel("Mode 1 Eurasian Core Loading (v1, Var1 = 26.0%)", fontsize=9.5)
ax1.set_ylabel("Mode 2 Maritime vs. Continental Divergence (v2, Var2 = 20.6%)", fontsize=9.5)
ax1.set_title("A. Structural Tri-Polarity of Eurasian Price Regimes (1739-1913)", fontsize=10.5, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = fig2.add_subplot(1, 2, 2)
ax2.plot(yrs_c, wr_df.loc[1738:1913, 'London'], color='#1f4e78', lw=1.7, label='London')
ax2.plot(yrs_c, wr_df.loc[1738:1913, 'Amsterdam'], color='#5b9bd5', lw=1.4, label='Amsterdam')
ax2.plot(yrs_c, wr_df.loc[1738:1913, 'Leipzig'], color='#385723', lw=1.3, linestyle='--', label='Leipzig')
ax2.plot(yrs_c, wr_df.loc[1738:1913, 'Beijing'], color='#c00000', lw=1.8, label='Beijing')
ax2.plot(yrs_c, wr_df.loc[1738:1913, 'Shanghai'], color='#ed7d31', lw=1.3, linestyle=':', label='Suzhou / Shanghai')
ax2.axhline(1.0, color='black', linestyle='--', lw=1.0, label='Barebones Subsistence Line (1.0)')
ax2.set_ylabel("Welfare Ratio (Subsistence Baskets per Year)", fontsize=9.5)
ax2.set_title("B. Real Living Standards Across Eurasia: The Great Divergence (1738-1913)", fontsize=10.5, fontweight='bold')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.4)

fig2.suptitle("Three Poles of Eurasia: Continental China, Maritime China, and Industrializing Europe", fontsize=12.5, fontweight='bold', y=1.01)
plt.savefig('Figure_2_Eurasia.png', bbox_inches='tight')
plt.close(fig2)
print("Clean Eurasian pipeline completed!")


=== TABLE 1: EURASIAN SPECTRAL SUMMARY ===
Unbroken 5-City CPI (1739-1913)        | M=175 | Var1=35.6% | Var2=21.7% | PR(u1)=0.264 [0.23, 0.31] | PR(v1)=0.668 [0.57, 0.75] | Break=1891 (p=0.0084)
Uninterpolated 7-City CPI (1739-1833)  | M= 95 | Var1=24.2% | Var2=20.2% | PR(u1)=0.352 [0.26, 0.39] | PR(v1)=0.647 [0.47, 0.76] | Break=1806 (p=0.1429)
Bridged 7-City CPI (1739-1913)         | M=175 | Var1=26.0% | Var2=20.6% | PR(u1)=0.292 [0.24, 0.33] | PR(v1)=0.587 [0.45, 0.77] | Break=1891 (p=0.0008)
Pre-Opium War CPI (1739-1839, N=5)     | M=101 | Var1=31.1% | Var2=24.7% | PR(u1)=0.262 [0.20, 0.38] | PR(v1)=0.569 [0.46, 0.74] | Break=1762 (p=0.3538)
Treaty Port & Silver CPI (1840-1913)   | M= 74 | Var1=59.2% | Var2=20.1% | PR(u1)=0.225 [0.19, 0.29] | PR(v1)=0.757 [0.72, 0.80] | Break=1899 (p=0.0737)
5-City Welfare Ratios (1739-1913)      | M=175 | Var1=31.2% | Var2=22.6% | PR(u1)=0.265 [0.22, 0.31] | PR(v1)=0.601 [0.48, 0.75] | Break=1857 (p=0.0000)
Clean Eurasian pipeline completed!
